In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('matches.csv')

In [5]:
columns_to_drop = [
    "match_type", "player_of_match","target_overs", "target_runs", 
    "umpire1", "umpire2","season", "city", "date", 
    "toss_winner", "toss_decision", "result", "result_margin", 
    "team1", "team2"
]

# Drop the columns that exist in the DataFrame
df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

In [7]:
team_name_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Deccan Chargers": "Sunrisers Hyderabad",
    "Gujarat Lions": "Gujarat Titans",
    "Kings XI Punjab": "Punjab Kings"
}

# Apply mapping to the 'winner' column
if "winner" in df.columns:
    df["winner"] = df["winner"].replace(team_name_mapping)

In [9]:
venue_mapping = {
    "M Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "M Chinnaswamy Stadium, Bengaluru": "M Chinnaswamy Stadium",

    "Punjab Cricket Association Stadium, Mohali": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium, Mohali": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium": "Punjab Cricket Association Stadium, Mohali",
    "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh": "Punjab Cricket Association Stadium, Mohali",

    "Wankhede Stadium": "Wankhede Stadium",
    "Wankhede Stadium, Mumbai": "Wankhede Stadium",

    "Eden Gardens": "Eden Gardens",
    "Eden Gardens, Kolkata": "Eden Gardens",

    "Sawai Mansingh Stadium": "Sawai Mansingh Stadium",
    "Sawai Mansingh Stadium, Jaipur": "Sawai Mansingh Stadium",

    "Rajiv Gandhi International Stadium, Uppal": "Rajiv Gandhi International Stadium, Hyderabad",
    "Rajiv Gandhi International Stadium, Uppal, Hyderabad": "Rajiv Gandhi International Stadium, Hyderabad",
    "Rajiv Gandhi International Stadium": "Rajiv Gandhi International Stadium, Hyderabad",

    "MA Chidambaram Stadium, Chepauk": "MA Chidambaram Stadium, Chepauk",
    "MA Chidambaram Stadium, Chepauk, Chennai": "MA Chidambaram Stadium, Chepauk",
    "MA Chidambaram Stadium": "MA Chidambaram Stadium, Chepauk",

    "Dr DY Patil Sports Academy": "Dr DY Patil Sports Academy",
    "Dr DY Patil Sports Academy, Mumbai": "Dr DY Patil Sports Academy",

    "Brabourne Stadium": "Brabourne Stadium",
    "Brabourne Stadium, Mumbai": "Brabourne Stadium",

    "Himachal Pradesh Cricket Association Stadium": "Himachal Pradesh Cricket Association Stadium",
    "Himachal Pradesh Cricket Association Stadium, Dharamsala": "Himachal Pradesh Cricket Association Stadium",

    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",

    "Subrata Roy Sahara Stadium": "Subrata Roy Sahara Stadium",

    "Maharashtra Cricket Association Stadium": "Maharashtra Cricket Association Stadium",
    "Maharashtra Cricket Association Stadium, Pune": "Maharashtra Cricket Association Stadium",


    "Arun Jaitley Stadium": "Arun Jaitley Stadium, Delhi",
    "Arun Jaitley Stadium, Delhi": "Arun Jaitley Stadium, Delhi",
    "Feroz Shah Kotla":"Arun Jaitley Stadium, Delhi",

}
df['venue_canonical'] = df['venue'].map(venue_mapping).fillna(df['venue'])
df = df[['id', 'winner', 'venue_canonical']]

In [11]:
df.rename(columns={"id": "match_id"}, inplace=True)

In [13]:
df.head()

,match_id,winner,venue_canonical
0,335982,Kolkata Knight Riders,M Chinnaswamy Stadium
1,335983,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali"
2,335984,Delhi Capitals,"Arun Jaitley Stadium, Delhi"
3,335985,Royal Challengers Bangalore,Wankhede Stadium
4,335986,Kolkata Knight Riders,Eden Gardens


In [15]:
df1 = pd.read_csv('deliveries.csv')

In [17]:
essential_columns = [
    "match_id", "inning", "batting_team", "bowling_team", 
    "over", "ball", "total_runs", "is_wicket"
]

# Subset the data (only keep the essential columns that exist in the dataset)
df1 = df1[[col for col in essential_columns if col in df1.columns]]

# Define the team name mapping
team_name_mapping = {
    "Delhi Daredevils": "Delhi Capitals",
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Deccan Chargers": "Sunrisers Hyderabad",
    "Gujarat Lions": "Gujarat Titans",
    "Kings XI Punjab": "Punjab Kings"
}

# Standardize team names (only if the columns exist)
for col in ["batting_team", "bowling_team"]:
    if col in df1.columns:
        df1[col] = df1[col].replace(team_name_mapping)

In [19]:
df1['cum_runs'] = df1.groupby(['match_id', 'inning'])['total_runs'].cumsum()
df1['cum_wickets'] = df1.groupby(['match_id', 'inning'])['is_wicket'].cumsum()
df1['overs_completed'] = df1['over'] + (df1['ball'] - 1) / 6
df1['current_run_rate'] = np.where(
    df1['overs_completed'] == 0,
    0,
   df1['cum_runs'] / df1['overs_completed']
)

In [21]:
df1.head()

,match_id,inning,batting_team,bowling_team,over,ball,total_runs,is_wicket,cum_runs,cum_wickets,overs_completed,current_run_rate
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,1,0,1,0,0.000000,0.0
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,0,0,1,0,0.166667,6.0
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,1,0,2,0,0.333333,6.0
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,0,0,2,0,0.500000,4.0
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,0,0,2,0,0.666667,3.0


In [23]:
first_innings = df1[df1['inning'] == 1]
first_innings_final = first_innings.groupby('match_id')['cum_runs'].max().reset_index()
first_innings_final = first_innings_final.rename(columns={'cum_runs': 'first_innings_score'})
first_innings_final['target'] = first_innings_final['first_innings_score'] + 1

In [25]:
final_data = pd.merge(df1, df, on='match_id', how='left')
final_data = pd.merge(final_data, first_innings_final[['match_id', 'target']], on='match_id', how='left')

In [27]:
final_data.head()

,match_id,inning,batting_team,bowling_team,over,ball,total_runs,is_wicket,cum_runs,cum_wickets,overs_completed,current_run_rate,winner,venue_canonical,target
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,1,0,1,0,0.000000,0.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,0,0,1,0,0.166667,6.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,1,0,2,0,0.333333,6.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,0,0,2,0,0.500000,4.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,0,0,2,0,0.666667,3.0,Kolkata Knight Riders,M Chinnaswamy Stadium,223


In [29]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Encode the 'winner' column
final_data["winner_encoded"] = label_encoder.fit_transform(final_data["winner"])

# Display unique encoded values for teams
print("Encoded Teams:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

Encoded Teams: {'Chennai Super Kings': 0, 'Delhi Capitals': 1, 'Gujarat Titans': 2, 'Kochi Tuskers Kerala': 3, 'Kolkata Knight Riders': 4, 'Lucknow Super Giants': 5, 'Mumbai Indians': 6, 'Pune Warriors': 7, 'Punjab Kings': 8, 'Rajasthan Royals': 9, 'Rising Pune Supergiants': 10, 'Royal Challengers Bangalore': 11, 'Royal Challengers Bengaluru': 12, 'Sunrisers Hyderabad': 13, nan: 14}


In [30]:
remaining_overs = 20 - final_data['overs_completed']
final_data['required_run_rate'] = np.where(
    (final_data['inning'] == 2) & (remaining_overs > 0),
    (final_data['target'] - final_data['cum_runs']) / remaining_overs,
    0
)
final_data['required_run_rate'] = final_data['required_run_rate'].replace([np.inf, -np.inf], 0)

# Create target variable: win = 1 if batting_team equals winner, else 0.
final_data['win'] = (final_data['batting_team'] == final_data['winner']).astype(int)

# Filter to second innings only (since required run rate applies to 2nd innings)
final_data = final_data[final_data['inning'] == 2].copy()

In [33]:
keep_cols = ['match_id', 'inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
             'required_run_rate', 'target', 'batting_team', 'bowling_team', 'winner_encoded', 'venue_canonical', 'win']
final_data = final_data[keep_cols]
final_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 125741 entries, 124 to 260919
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   match_id           125741 non-null  int64  
 1   inning             125741 non-null  int64  
 2   cum_runs           125741 non-null  int64  
 3   cum_wickets        125741 non-null  int64  
 4   current_run_rate   125741 non-null  float64
 5   required_run_rate  125741 non-null  float64
 6   target             125741 non-null  int64  
 7   batting_team       125741 non-null  object 
 8   bowling_team       125741 non-null  object 
 9   winner_encoded     125741 non-null  int32  
 10  venue_canonical    125741 non-null  object 
 11  win                125741 non-null  int32  
dtypes: float64(2), int32(2), int64(5), object(3)
memory usage: 11.5+ MB


In [37]:
final_data.head()

,match_id,inning,cum_runs,cum_wickets,current_run_rate,required_run_rate,target,batting_team,bowling_team,winner_encoded,venue_canonical,win
124,335982,2,1,0,0.0,11.100000,223,Royal Challengers Bangalore,Kolkata Knight Riders,4,M Chinnaswamy Stadium,0
125,335982,2,2,0,12.0,11.142857,223,Royal Challengers Bangalore,Kolkata Knight Riders,4,M Chinnaswamy Stadium,0
126,335982,2,2,0,6.0,11.237288,223,Royal Challengers Bangalore,Kolkata Knight Riders,4,M Chinnaswamy Stadium,0
127,335982,2,3,0,6.0,11.282051,223,Royal Challengers Bangalore,Kolkata Knight Riders,4,M Chinnaswamy Stadium,0
128,335982,2,4,0,6.0,11.327586,223,Royal Challengers Bangalore,Kolkata Knight Riders,4,M Chinnaswamy Stadium,0


In [65]:
from sklearn.preprocessing import LabelEncoder

# Define categorical columns
categorical_cols = ['batting_team', 'bowling_team', 'venue_canonical']

# Apply Label Encoding
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    final_data[col] = le.fit_transform(final_data[col])
    label_encoders[col] = le  # Store the encoders for later use

# Now final_data is fully numerical and ready for training


In [87]:
final_data.head()

,match_id,inning,cum_runs,cum_wickets,current_run_rate,required_run_rate,target,batting_team,bowling_team,winner_encoded,venue_canonical,win
124,335982,2,1,0,0.0,11.100000,223,11,4,4,16,0
125,335982,2,2,0,12.0,11.142857,223,11,4,4,16,0
126,335982,2,2,0,6.0,11.237288,223,11,4,4,16,0
127,335982,2,3,0,6.0,11.282051,223,11,4,4,16,0
128,335982,2,4,0,6.0,11.327586,223,11,4,4,16,0
